In [18]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Project root
PROJECT_ROOT = Path.cwd().parent

print("FINAL ML MODEL VALIDATION")
print("=" * 50)

print("Project Root:")
print(PROJECT_ROOT.resolve())

print("\nModels folder:")
print((PROJECT_ROOT / "models").resolve())

print("\nReports folder:")
print((PROJECT_ROOT / "reports").resolve())

FINAL ML MODEL VALIDATION
Project Root:
C:\Users\user\Documents\Zidio_Project\FORESIGHT

Models folder:
C:\Users\user\Documents\Zidio_Project\FORESIGHT\models

Reports folder:
C:\Users\user\Documents\Zidio_Project\FORESIGHT\reports


In [2]:
# Step 30: Inspect Existing ML Models and Reports

print("EXISTING ML FILES")
print("=" * 50)

models_path = PROJECT_ROOT / "models"
reports_path = PROJECT_ROOT / "reports"

print("\nMODEL FILES:")
if models_path.exists():
    model_files = list(models_path.iterdir())

    if model_files:
        for file in sorted(model_files):
            print("-", file.name)
    else:
        print("No files found.")
else:
    print("Models folder not found.")

print("\nREPORT FILES:")
if reports_path.exists():
    report_files = list(reports_path.iterdir())

    if report_files:
        for file in sorted(report_files):
            print("-", file.name)
    else:
        print("No files found.")
else:
    print("Reports folder not found.")

EXISTING ML FILES

MODEL FILES:
- baseline_random_forest.pkl

REPORT FILES:
- baseline_predictions.csv
- model_evaluation_metrics.csv
- risk_scoring.csv


In [3]:
# Step 31: Load Final Dataset and Saved Model

print("LOADING FINAL DATASET AND SAVED MODEL")
print("=" * 50)

# Final validated dataset
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_engineered_dataset_final.csv"
)

# Saved model
MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "baseline_random_forest.pkl"
)

# Load dataset
final_df = pd.read_csv(DATA_PATH)

# Convert date columns
final_df["date"] = pd.to_datetime(final_df["date"])
final_df["launch_date"] = pd.to_datetime(final_df["launch_date"])

# Load model
model = joblib.load(MODEL_PATH)

print("Dataset loaded successfully!")
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))

print("\nModel loaded successfully!")
print("Model type:", type(model).__name__)

print("\nModel features:")
print(model.feature_names_in_)

LOADING FINAL DATASET AND SAVED MODEL
Dataset loaded successfully!
Rows: 73000
Columns: 24

Model loaded successfully!
Model type: RandomForestRegressor

Model features:
['unit_price' 'promo_flag' 'unit_cost' 'list_price' 'week' 'month'
 'is_holiday' 'day' 'day_of_week' 'quarter' 'year' 'on_hand_units'
 'on_order_units' 'lead_time_days' 'reorder_point']


In [4]:
# Step 32: Verify Model Feature Compatibility

print("MODEL FEATURE COMPATIBILITY")
print("=" * 50)

model_features = list(model.feature_names_in_)

required_features = [
    "unit_price",
    "promo_flag",
    "unit_cost",
    "list_price",
    "week",
    "month",
    "is_holiday",
    "day",
    "day_of_week",
    "quarter",
    "year",
    "on_hand_units",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]

missing_features = [
    col for col in model_features
    if col not in final_df.columns
]

extra_features = [
    col for col in required_features
    if col not in model_features
]

print("Model expects:", len(model_features), "features")
print("Required features:", len(required_features))

print("\nMissing model features from dataset:")
print(missing_features)

print("\nUnexpected feature differences:")
print(extra_features)

if model_features == required_features and not missing_features and not extra_features:
    print("\nPASS: Model and final dataset features are fully compatible.")
else:
    print("\nFAIL: Feature compatibility issue found.")

MODEL FEATURE COMPATIBILITY
Model expects: 15 features
Required features: 15

Missing model features from dataset:
[]

Unexpected feature differences:
[]

PASS: Model and final dataset features are fully compatible.


In [5]:
# Step 33: Generate Final Model Predictions

print("GENERATING FINAL MODEL PREDICTIONS")
print("=" * 50)

feature_columns = list(model.feature_names_in_)

X_final = final_df[feature_columns].copy()

# Safety check for missing values in model features
missing_model_values = X_final.isnull().sum().sum()

print("Rows used for prediction:", len(X_final))
print("Features used:", len(feature_columns))
print("Missing values in model features:", missing_model_values)

if missing_model_values > 0:
    print("\nFAIL: Missing values found in model features.")
else:
    # Generate predictions
    final_predictions = model.predict(X_final)

    final_df["Final_Predicted_Units_Sold"] = final_predictions

    print("\nPredictions generated successfully!")
    print("Prediction rows:", len(final_predictions))

    print("\nPrediction statistics:")
    print("Minimum:", final_predictions.min())
    print("Maximum:", final_predictions.max())
    print("Mean:", final_predictions.mean())
    print("Median:", np.median(final_predictions))

    print("\nFirst 10 predictions:")
    print(final_df[
        ["date", "sku_id", "units_sold", "Final_Predicted_Units_Sold"]
    ].head(10))

    print("\nPASS: Final predictions generated successfully.")

GENERATING FINAL MODEL PREDICTIONS
Rows used for prediction: 73000
Features used: 15
Missing values in model features: 0

Predictions generated successfully!
Prediction rows: 73000

Prediction statistics:
Minimum: 3.14
Maximum: 24.43
Mean: 11.934772465753424
Median: 11.68

First 10 predictions:
        date  sku_id  units_sold  Final_Predicted_Units_Sold
0 2024-01-01  SKU001          16                       14.24
1 2024-01-01  SKU002           9                       10.11
2 2024-01-01  SKU003          14                       12.40
3 2024-01-01  SKU004          11                       10.72
4 2024-01-01  SKU005          12                       10.77
5 2024-01-01  SKU006          18                       15.34
6 2024-01-01  SKU007          14                       11.91
7 2024-01-01  SKU008           5                        7.35
8 2024-01-01  SKU009          11                       13.32
9 2024-01-01  SKU010          15                       15.89

PASS: Final predictions generate

In [6]:
# Step 34: Calculate Final ML Model Metrics

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("FINAL ML MODEL PERFORMANCE")
print("=" * 50)

y_actual = final_df["units_sold"]
y_predicted = final_df["Final_Predicted_Units_Sold"]

mae = mean_absolute_error(y_actual, y_predicted)
rmse = np.sqrt(mean_squared_error(y_actual, y_predicted))
r2 = r2_score(y_actual, y_predicted)

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

print("\nActual Units Sold:")
print("Mean:", round(y_actual.mean(), 4))
print("Min :", y_actual.min())
print("Max :", y_actual.max())

print("\nPredicted Units Sold:")
print("Mean:", round(y_predicted.mean(), 4))
print("Min :", round(y_predicted.min(), 4))
print("Max :", round(y_predicted.max(), 4))

print("\nPASS: Final model metrics calculated successfully.")

FINAL ML MODEL PERFORMANCE
MAE : 2.7793
RMSE: 3.5746
R²  : 0.2847

Actual Units Sold:
Mean: 11.9012
Min : 0
Max : 32

Predicted Units Sold:
Mean: 11.9348
Min : 3.14
Max : 24.43

PASS: Final model metrics calculated successfully.


In [7]:
# Step 35: Compare Final Metrics With Existing Evaluation Metrics

print("COMPARING FINAL AND EXISTING MODEL METRICS")
print("=" * 50)

metrics_path = PROJECT_ROOT / "reports" / "model_evaluation_metrics.csv"

if metrics_path.exists():

    existing_metrics = pd.read_csv(metrics_path)

    print("\nExisting evaluation metrics:")
    print(existing_metrics.to_string(index=False))

    print("\nFinal validation metrics:")
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²  :", round(r2, 4))

    print("\nPASS: Existing evaluation metrics loaded successfully.")

else:
    print("FAIL: model_evaluation_metrics.csv not found.")

COMPARING FINAL AND EXISTING MODEL METRICS

Existing evaluation metrics:
  Metric     Value
     MAE  1.678682
    RMSE  2.360417
R2 Score  0.688088
MAPE (%) 21.374626

Final validation metrics:
MAE : 2.7793
RMSE: 3.5746
R²  : 0.2847

PASS: Existing evaluation metrics loaded successfully.


In [8]:
# Step 36: Validate Existing Prediction File

print("EXISTING PREDICTION VALIDATION")
print("=" * 50)

predictions_path = PROJECT_ROOT / "reports" / "baseline_predictions.csv"

existing_predictions = pd.read_csv(predictions_path)

print("\nPrediction file shape:")
print("Rows:", len(existing_predictions))
print("Columns:", len(existing_predictions.columns))

print("\nColumns:")
print(existing_predictions.columns.tolist())

print("\nFirst 10 rows:")
print(existing_predictions.head(10))

print("\nMissing values:")
print(existing_predictions.isnull().sum())

print("\nNumeric summary:")
print(existing_predictions.describe())

EXISTING PREDICTION VALIDATION


C:\Users\user\AppData\Local\Temp\ipykernel_12824\1290491275.py:8: DtypeWarning: Columns (0: promo_event) have mixed types. Specify dtype option on import or set low_memory=False.
  existing_predictions = pd.read_csv(predictions_path)



Prediction file shape:
Rows: 73000
Columns: 25

Columns:
['date', 'sku_id', 'units_sold', 'revenue', 'unit_price', 'promo_flag', 'category', 'subcategory', 'launch_date', 'unit_cost', 'list_price', 'week', 'month', 'season', 'is_holiday', 'promo_event', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point', 'day', 'day_of_week', 'quarter', 'year', 'Predicted_Units_Sold']

First 10 rows:
         date  sku_id  units_sold  revenue  unit_price  promo_flag  \
0  2024-01-01  SKU001          16  3814.08      238.38           0   
1  2024-01-01  SKU002           9   943.65      104.85           0   
2  2024-01-01  SKU003          14  4658.92      332.78           0   
3  2024-01-01  SKU004          11  2825.35      256.85           0   
4  2024-01-01  SKU005          12   575.40       47.95           0   
5  2024-01-01  SKU006          18  3963.60      220.20           0   
6  2024-01-01  SKU007          14  1491.70      106.55           0   
7  2024-01-01  SKU008           5 

In [9]:
# Step 37: Validate Existing Prediction Accuracy

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("EXISTING PREDICTION ACCURACY CHECK")
print("=" * 50)

actual_existing = existing_predictions["units_sold"]
pred_existing = existing_predictions["Predicted_Units_Sold"]

existing_mae = mean_absolute_error(actual_existing, pred_existing)
existing_rmse = np.sqrt(mean_squared_error(actual_existing, pred_existing))
existing_r2 = r2_score(actual_existing, pred_existing)

print("Rows:", len(existing_predictions))

print("\nCalculated metrics from baseline_predictions.csv:")
print("MAE :", round(existing_mae, 4))
print("RMSE:", round(existing_rmse, 4))
print("R²  :", round(existing_r2, 4))

print("\nMetrics stored in model_evaluation_metrics.csv:")
print("MAE :", 1.678682)
print("RMSE:", 2.360417)
print("R²  :", 0.688088)

print("\nDifference:")
print("MAE difference :", round(existing_mae - 1.678682, 4))
print("RMSE difference:", round(existing_rmse - 2.360417, 4))
print("R² difference  :", round(existing_r2 - 0.688088, 4))

EXISTING PREDICTION ACCURACY CHECK
Rows: 73000

Calculated metrics from baseline_predictions.csv:
MAE : 1.6787
RMSE: 2.3604
R²  : 0.6881

Metrics stored in model_evaluation_metrics.csv:
MAE : 1.678682
RMSE: 2.360417
R²  : 0.688088

Difference:
MAE difference : -0.0
RMSE difference: 0.0
R² difference  : 0.0


In [10]:
# Step 38: Final Model Validation Comparison

print("FINAL MODEL VALIDATION COMPARISON")
print("=" * 50)

print("\nOriginal Model Evaluation:")
print("MAE :", round(existing_mae, 4))
print("RMSE:", round(existing_rmse, 4))
print("R²  :", round(existing_r2, 4))

print("\nCorrected Final Dataset Evaluation:")
print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

print("\nPerformance Change:")

mae_change = mae - existing_mae
rmse_change = rmse - existing_rmse
r2_change = r2 - existing_r2

print("MAE change :", round(mae_change, 4))
print("RMSE change:", round(rmse_change, 4))
print("R² change  :", round(r2_change, 4))

print("\nPrediction Coverage:")
print("Actual rows:", len(final_df))
print("Predicted rows:", len(final_df[final_df["Final_Predicted_Units_Sold"].notna()]))

if len(final_df) == len(final_df[final_df["Final_Predicted_Units_Sold"].notna()]):
    print("All final rows have predictions.")
else:
    print("Some final rows are missing predictions.")

FINAL MODEL VALIDATION COMPARISON

Original Model Evaluation:
MAE : 1.6787
RMSE: 2.3604
R²  : 0.6881

Corrected Final Dataset Evaluation:
MAE : 2.7793
RMSE: 3.5746
R²  : 0.2847

Performance Change:
MAE change : 1.1006
RMSE change: 1.2142
R² change  : -0.4034

Prediction Coverage:
Actual rows: 73000
Predicted rows: 73000
All final rows have predictions.


In [13]:
# Step 39: Train Final Random Forest Model - Faster Validation Version

print("TRAINING FINAL RANDOM FOREST MODEL")
print("=" * 50)

feature_columns = [
    "unit_price",
    "promo_flag",
    "unit_cost",
    "list_price",
    "week",
    "month",
    "is_holiday",
    "day",
    "day_of_week",
    "quarter",
    "year",
    "on_hand_units",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]

target_column = "units_sold"

model_df = final_df.sort_values("date").reset_index(drop=True)

split_index = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining date range:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTesting date range:")
print(test_df["date"].min(), "to", test_df["date"].max())

# Faster validation model
final_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("\nStarting model training...")

final_model.fit(X_train, y_train)

print("Model training completed.")

print("\nModel type:", type(final_model).__name__)
print("Number of trees:", final_model.n_estimators)

TRAINING FINAL RANDOM FOREST MODEL
Training rows: 58400
Testing rows: 14600

Training date range:
2024-01-01 00:00:00 to 2024-10-18 00:00:00

Testing date range:
2024-10-19 00:00:00 to 2024-12-30 00:00:00

Starting model training...
Model training completed.

Model type: RandomForestRegressor
Number of trees: 50


In [14]:
# Step 40: Evaluate Final Random Forest on Test Data

print("FINAL RANDOM FOREST TEST EVALUATION")
print("=" * 50)

test_predictions = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, test_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
final_r2 = r2_score(y_test, test_predictions)

print("Test rows:", len(y_test))
print("Prediction rows:", len(test_predictions))

print("\nFinal Test Metrics:")
print("MAE :", round(final_mae, 4))
print("RMSE:", round(final_rmse, 4))
print("R²  :", round(final_r2, 4))

FINAL RANDOM FOREST TEST EVALUATION
Test rows: 14600
Prediction rows: 14600

Final Test Metrics:
MAE : 3.2449
RMSE: 4.0551
R²  : 0.0946


In [15]:
# Step 41: Final Model Quality Check

print("FINAL MODEL QUALITY CHECK")
print("=" * 50)

print("Test rows:", len(y_test))
print("Predictions:", len(test_predictions))

print("\nFinal Test Metrics:")
print("MAE :", round(final_mae, 4))
print("RMSE:", round(final_rmse, 4))
print("R²  :", round(final_r2, 4))

print("\nPrediction Range:")
print("Minimum:", round(test_predictions.min(), 4))
print("Maximum:", round(test_predictions.max(), 4))
print("Mean:", round(test_predictions.mean(), 4))

print("\nActual Range:")
print("Minimum:", y_test.min())
print("Maximum:", y_test.max())
print("Mean:", round(y_test.mean(), 4))

print("\nPrediction Coverage:")
print("Missing predictions:", np.isnan(test_predictions).sum())

if (
    len(test_predictions) == len(y_test)
    and np.isnan(test_predictions).sum() == 0
):
    print("\nFinal model output is complete and valid.")
else:
    print("\nFinal model output requires investigation.")

FINAL MODEL QUALITY CHECK
Test rows: 14600
Predictions: 14600

Final Test Metrics:
MAE : 3.2449
RMSE: 4.0551
R²  : 0.0946

Prediction Range:
Minimum: 9.0169
Maximum: 18.9881
Mean: 11.8923

Actual Range:
Minimum: 0
Maximum: 31
Mean: 11.9153

Prediction Coverage:
Missing predictions: 0

Final model output is complete and valid.


In [16]:
# Step 42: Save Final Model and Validation Metrics

print("SAVING FINAL ML MODEL")
print("=" * 50)

final_model_path = PROJECT_ROOT / "models" / "final_random_forest.pkl"
final_metrics_path = PROJECT_ROOT / "reports" / "final_model_validation_metrics.csv"

# Save model
joblib.dump(final_model, final_model_path)

# Save test metrics
final_metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [final_mae, final_rmse, final_r2]
})

final_metrics.to_csv(final_metrics_path, index=False)

print("Model saved to:")
print(final_model_path.resolve())

print("\nMetrics saved to:")
print(final_metrics_path.resolve())

print("\nModel file exists:", final_model_path.exists())
print("Metrics file exists:", final_metrics_path.exists())

SAVING FINAL ML MODEL
Model saved to:
C:\Users\user\Documents\Zidio_Project\FORESIGHT\models\final_random_forest.pkl

Metrics saved to:
C:\Users\user\Documents\Zidio_Project\FORESIGHT\reports\final_model_validation_metrics.csv

Model file exists: True
Metrics file exists: True


In [17]:
# Step 43: Final ML Model Validation Verification

print("FINAL ML MODEL VALIDATION")
print("=" * 50)

# Reload saved model
validated_model = joblib.load(final_model_path)

# Reload saved metrics
validated_metrics = pd.read_csv(final_metrics_path)

print("Saved model exists:", final_model_path.exists())
print("Saved metrics exists:", final_metrics_path.exists())

print("\nModel type:")
print(type(validated_model).__name__)

print("\nSaved metrics:")
print(validated_metrics.to_string(index=False))

print("\nTest rows:", len(y_test))
print("Predictions:", len(test_predictions))

print("\nFinal validation results:")
print("MAE :", round(final_mae, 4))
print("RMSE:", round(final_rmse, 4))
print("R²  :", round(final_r2, 4))

print("\nModel features:", len(validated_model.feature_names_in_))
print("Expected features:", len(feature_columns))

print("\nPrediction missing values:", np.isnan(test_predictions).sum())

FINAL ML MODEL VALIDATION
Saved model exists: True
Saved metrics exists: True

Model type:
RandomForestRegressor

Saved metrics:
Metric    Value
   MAE 3.244873
  RMSE 4.055057
    R2 0.094567

Test rows: 14600
Predictions: 14600

Final validation results:
MAE : 3.2449
RMSE: 4.0551
R²  : 0.0946

Model features: 15
Expected features: 15

Prediction missing values: 0
